# 02 – Bee Haven: Datenqualität (N2)

**Ziel:** Fachlich unplausible Werte markieren und flow in Ausflug/Einflug trennen; Ergebnis mit UTC-Zeitstempel nach `silver/cleaned` speichern.

**Regeln (Pflichtenheft PH-ITS-002, Kap. 2.2):** R5 weight < 20 kg → ungültig · R5a Zeitraum 28.04.–02.07.2018 „Stock nicht auf Waage“ (E5) → ungültig · R6 humidity < 0 % → ungültig · R7 temperature < 0 °C behalten · R8 flow in 2 Spalten trennen · R9 Ausreißer (robuster z-Score) markieren

**Grundsatz:** markieren statt löschen – ungültige Werte werden leer (NaN), die Spalte `qualitaet` nennt den Grund; statistische Ausreißer (R9) bekommen eine eigene Spalte `<messwert>_auffaellig` und behalten ihren Wert.

> **Synapse-Version (Azure Synapse Analytics)** – automatisch erzeugt aus `../02_qualitaet.ipynb` durch
> `Privat/03_notebooks_ITS-Data/generator/make_synapse_notebooks.py`. **Nicht von Hand ändern**, sondern das lokale
> Notebook ändern und den Generator erneut ausführen.
>
> Unterschiede zur lokalen Version (Vorbild: Kurs-Notebook 04 „weather pipeline synapse ready“):
> - Pfade als **ABFSS-URI (Azure Blob File System)** `abfss://<container>@<konto>.dfs.core.windows.net/...` statt `pathlib.Path`.
> - **Parameterzelle (Parameter Cell)** direkt darunter: eine Pipeline (Notebook-Aktivität) kann die Werte überschreiben.
> - **Kein Kontoschlüssel (Account Key) im Code:** Zugriff über die Identität des Synapse-Arbeitsbereichs (Workspace Identity)
>   bzw. einen verknüpften Dienst (Linked Service). Das Kurs-Notebook trägt den Schlüssel im Klartext ein – Sicherheitsrisiko.
> - Diagramme werden angezeigt; als PNG in den Lake nur bei `BILDER_SPEICHERN = True`.
> - Kosten: Ausführung nur nach Freigabe (Spark-Pool kostet je Minute).

In [ ]:
# --- Parameter (Parameters) – in Synapse als Parameterzelle markiert; Pipeline kann Werte überschreiben ---
STORAGE_ACCOUNT = "mabeehavenlakehouse"   # Speicherkonto (Storage Account, ADLS Gen2) des Lakehouse
CONTAINER = "lakehouse"                   # Container mit bronze/, silver/, gold/
STADT = "schwartau"                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen
LINKED_SERVICE = ""                       # leer = Identität des Arbeitsbereichs; sonst Name des verknüpften Dienstes (Linked Service)
BILDER_SPEICHERN = False                  # Diagramme zusätzlich als PNG nach gold/<stadt>/bilder/ schreiben?
LOKALER_TEST = ""                            # leer = Azure; lokaler Ordner (Spiegel des Containers) = Test ohne Azure

In [ ]:
# --- Hilfsfunktionen für Dateizugriffe: Azure Data Lake (abfss) oder lokaler Testordner ---
import os                                 # lokale Ordner (nur Testmodus)
import fnmatch                            # Dateinamen mit Platzhalter * vergleichen
import tempfile                           # Zwischenablage für PNG-Dateien

AZURE = not LOKALER_TEST                                                   # True = in Synapse gegen den Data Lake arbeiten
WURZEL = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"   # ABFSS-Wurzel des Containers (wie Kurs-Notebook 04)
          if AZURE else LOKALER_TEST.rstrip("/") + "/")                    # oder lokaler Testordner
SO = ({"linked_service": LINKED_SERVICE} if LINKED_SERVICE else {}) if AZURE else None   # storage_options für pandas
# Azure: SO nie None – sonst versucht pandas bei Parquet zuerst pyarrow direkt mit abfss:// und bricht ab
# ("Arrow compiled without Azure Blob File System support"); mit {} nimmt pandas den Synapse-Zugang über fsspec.
if AZURE:
    from notebookutils import mssparkutils                                 # Synapse-Werkzeuge: Dateien auflisten, Ordner anlegen, kopieren

def pfad(*teile):
    """Pfad unterhalb der Wurzel zusammensetzen, z. B. pfad("silver", "cleaned") -> abfss://.../silver/cleaned"""
    return WURZEL + "/".join(str(t).strip("/") for t in teile)

def dateien(ordner, muster="*"):
    """Sortierte Liste (voller Pfad, Größe in Byte) aller Dateien in EINEM Ordner, deren Name zum Muster passt."""
    if AZURE:
        try:
            eintraege = [(f.path, f.size) for f in mssparkutils.fs.ls(ordner) if f.isFile and fnmatch.fnmatch(f.name, muster)]
        except Exception:                                                  # Ordner existiert (noch) nicht
            eintraege = []
    else:
        eintraege = [(os.path.join(ordner, n), os.path.getsize(os.path.join(ordner, n)))
                     for n in (os.listdir(ordner) if os.path.isdir(ordner) else []) if fnmatch.fnmatch(n, muster)]
    return sorted(eintraege)                                               # sortiert nach Name -> letzte = neueste Version (Zeitstempel)

def neueste(ordner, muster):
    """Pfad der jüngsten Datei (Name enthält UTC-Zeitstempel) – Fehler, wenn keine vorhanden."""
    treffer = dateien(ordner, muster)
    if not treffer:
        raise FileNotFoundError(f"keine Datei {muster} in {ordner}")
    return treffer[-1][0]

def ordner_anlegen(ordner):
    """Zielordner anlegen, falls nicht vorhanden (in ADLS Gen2 unschädlich, wenn schon da)."""
    mssparkutils.fs.mkdirs(ordner) if AZURE else os.makedirs(ordner, exist_ok=True)

def archivieren(datei, neuer_name=None):
    """Datei in das _archiv ihres eigenen Ordners verschieben (Regel: jeder Ordner hat ein _archiv) – nichts wird gelöscht."""
    ordner, name = datei.rsplit("/", 1)
    ziel = f"{ordner}/_archiv/{neuer_name or name}"
    ordner_anlegen(f"{ordner}/_archiv")
    mssparkutils.fs.mv(datei, ziel) if AZURE else os.replace(datei, ziel)

def groesse_mb(datei):
    """Dateigröße in Megabyte (zur Kontrolle nach dem Speichern)."""
    ordner, name = datei.rsplit("/", 1)
    return next((g for p, g in dateien(ordner, name)), 0) / 1_000_000

def bild_speichern(fig, name, **optionen):
    """Diagramm optional als PNG in den Lake schreiben (Parameter BILDER_SPEICHERN)."""
    if not BILDER_SPEICHERN:
        return
    ziel_ordner = pfad("gold", STADT, "bilder")
    ordner_anlegen(ziel_ordner)
    lokal = os.path.join(tempfile.gettempdir(), name)                      # erst lokal auf dem Knoten speichern ...
    fig.savefig(lokal, **optionen)
    if AZURE:
        mssparkutils.fs.cp("file:" + lokal, ziel_ordner + "/" + name)       # ... dann in den Data Lake kopieren
    else:
        os.replace(lokal, os.path.join(ziel_ordner, name))

print("Wurzel:", WURZEL, "| Azure:", AZURE)

## 0. Bibliotheken und Pfade

**Kontrollwert:** keine Ausgabe, keine Fehlermeldung

In [ ]:
# --- Bibliotheken laden ---
import pandas as pd                      # pandas: Tabellen einlesen, prüfen, speichern
from datetime import datetime, timezone  # für den UTC-Zeitstempel im Dateinamen

# --- Pfade festlegen ---
ZWISCHENSTAND = pfad("silver", "zwischenstand", STADT)         # Eingang: Ergebnis aus N1
CLEANED = pfad("silver", "cleaned", STADT)                     # Ausgang: bereinigte Daten
sensoren = ["flow", "humidity", "temperature", "weight"]       # Liste der Sensoren

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(pfad("konfiguration", "standorte.csv"), index_col="stadt", storage_options=SO)
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
sensoren_cfg = pd.read_csv(pfad("konfiguration", "sensoren.csv"), storage_options=SO)          # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(pfad("konfiguration", "ausschluesse.csv"), storage_options=SO)      # bekannte Ausfallzeiträume
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit

**Ergebnis:**

Bibliotheken und Pfade ohne Fehler geladen.

## 1. Import der N1-Ergebnisse

**Kontrollwert (Ergebnis N1, muss übereinstimmen):** 4 Tabellen mit den Spalten `timestamp`, Messwert, `zeit_utc` (Typ UTC-Zeit); flow 2.513.836 · humidity 1.761 · temperature 253.430 · weight 1.761 Zeilen

In [ ]:
# --- N1-Ergebnisse (Parquet) einlesen ---
n1 = {}                                                                   # leeres Dictionary: Sensor -> Tabelle
for s in sensoren:                                                        # für jeden Sensor ...
    n1[s] = pd.read_parquet(f"{ZWISCHENSTAND}/{s}_{STADT}_n1.parquet", storage_options=SO)  # ... Zwischenstand aus N1 laden
    print(f"{s:12} {len(n1[s]):>10,} Zeilen  {dict(n1[s].dtypes.astype(str))}")  # ... Zeilen + Datentypen prüfen

**Ergebnis:**

flow 2.513.836 · humidity 1.761 · temperature 253.430 · weight 1.761 Zeilen – wie in N1.

`zeit_utc` ist weiterhin `datetime64[us, UTC]` → Parquet hat die Datentypen behalten.

## 2. Arbeitskopien erstellen

**Kontrollwert:** je Sensor eine Kopie mit neuer Spalte `qualitaet` („ok“ bzw. „fehlend“)

In [ ]:
# --- Arbeitskopien: N1-Stand bleibt zum Vergleich unverändert ---
q = {s: tabelle.copy() for s, tabelle in n1.items()}   # q = "quality" (Namenskonvention wie in der Kursvorlage)
for s in sensoren:                                     # für jeden Sensor ...
    q[s]["qualitaet"] = "ok"                           # ... zunächst alle Werte als "ok" markieren
    q[s].loc[q[s][s].isna(), "qualitaet"] = "fehlend"  # ... schon fehlende Werte (aus N1, R3) als "fehlend" markieren
    print(f"{s:12}", q[s]["qualitaet"].value_counts().to_dict())  # ... Kontrolle

**Ergebnis:**

Spalte `qualitaet` angelegt: flow 2.513.836 ok · humidity 1.749 ok / 12 fehlend · temperature 251.398 ok / 2.032 fehlend · weight 1.749 ok / 12 fehlend.

## 3. weight – Messfehler markieren (R5)

**Regel:** weight < 20.000 g (auch negative Werte) → ungültig

**Warum?** Ein Stock mit Beute wiegt realistisch 30–90 kg; Werte darunter sind Sensorfehler und würden Tagesdurchschnitte zerstören.

**Kontrollwert (Voranalyse, Pflichtenheft R5):** 102 ungültig (darin die 37 negativen), 12 fehlend, 1.647 ok

**Eigene Erwartung (fachlich noch zu belegen):** Ein bewohnter Stock wiegt mit Beute realistisch ca. 30–90 kg; negative Gewichte sind physikalisch unmöglich.

In [ ]:
# --- R5: weight unter der Mindestgrenze (konfiguration/sensoren.csv, in kg) als ungültig markieren ---
grenze_kg = REGELN.loc["weight", "min"]                             # z. B. 20 kg
ungueltig = q["weight"]["weight"] < grenze_kg * 1000                # True, wo das Gewicht (intern Gramm) darunter liegt
q["weight"].loc[ungueltig, "qualitaet"] = f"ungueltig: < {grenze_kg:g} kg"   # Grund in der Spalte qualitaet vermerken
q["weight"].loc[ungueltig, "weight"] = float("nan")                 # Messwert leeren (markieren statt Zeile löschen)
q["weight"]["qualitaet"].value_counts()                             # Kontrolle: Anzahl je Qualitätsstufe


**Ergebnis:**

weight: 1.647 ok · **102 ungültig (< 20 kg)** · 12 fehlend – wie erwartet (R5).

## 3a. weight – Stock nicht auf der Waage (R5a, Befund E5)

**Befund E5:** Vom 28.04.2018 (mittags) bis 02.07.2018 (morgens) zeigt die Waage zuerst ca. −0,17 kg (leere Waage), dann steigende Werte bis 25,6 kg, dazwischen einen Abfall auf 1,2 kg und am Ende wieder ca. −0,15 kg. Davor wog der Stock gleichmäßig ca. 48,6 kg, danach 57–66 kg. R5 (< 20 kg) erfasst nur 102 dieser 130 Werte; 28 Werte zwischen 20,3 und 25,6 kg blieben bisher „ok“.

**Mögliche Ursachen:** (a) Eingriff der Imkerei – Stock von der Waage genommen oder umgesetzt · (b) Sensorstörung. Eine reine Honigentnahme erklärt den Verlauf nicht: Kasten, Rähmchen, Bienen und Brut bleiben auf der Waage, das Gewicht fiele nicht auf 0 kg. Ohne Imkerei-Protokoll ist (a) oder (b) nicht entscheidbar.

**Lösungswege:** A – ganzen Zeitraum markieren · B – Grenze auf 30 kg anheben (würde auch einen echten, leichten Stock treffen, z. B. ein junges Volk).

**Entscheidung (Projektleitung 30.09.2026): Weg A** – alle Werte im Zeitraum werden ungültig markiert (Wert leer, Grund in `qualitaet`); die Grenze von R5 bleibt 20 kg.

**Kontrollwert:** 130 Werte im Zeitraum, davon 102 schon durch R5 markiert → 28 zusätzlich ungültig; danach 1.619 ok · 130 ungültig · 12 fehlend.


In [ ]:
# --- R5a: bekannte Ausfallzeiträume (konfiguration/ausschluesse.csv, z. B. Schwartau E5 „Stock nicht auf Waage“) als ungültig markieren ---
w = q["weight"]                                                     # kurzer Name für die weight-Tabelle
for _, a in AUSSCHLUESSE[AUSSCHLUESSE["sensor"] == "weight"].iterrows():   # je eingetragenem Zeitraum dieser Stadt ...
    im_zeitraum = w["zeit_utc"].between(a["von_utc"], a["bis_utc"])  # ... True für alle Messungen im Zeitraum (inkl. Grenzen)
    betroffen = im_zeitraum & (w["qualitaet"] != "fehlend")         # fehlende Werte behalten ihren Grund "fehlend"
    print(a["grund"], "| im Zeitraum:", im_zeitraum.sum(), "| davon bisher ok:", (betroffen & (w["qualitaet"] == "ok")).sum())  # Kontrolle (Schwartau: 130 | 28)
    w.loc[betroffen, "qualitaet"] = f"ungueltig: {a['grund']}"      # einheitlicher Grund für den ganzen Zeitraum
    w.loc[betroffen, "weight"] = float("nan")                       # Messwert leeren (markieren statt Zeile löschen)
w["qualitaet"].value_counts()                                       # Kontrolle: Anzahl je Qualitätsstufe


**Ergebnis:** siehe Ausgabe – erwartet 28 zusätzlich ungültig; weight danach 1.619 ok · 130 ungültig (Zeitraum E5 bzw. < 20 kg) · 12 fehlend.


## 4. humidity – Fehlercodes markieren (R6)

**Regel:** humidity < 0 % → ungültig

**Warum?** Luftfeuchte liegt physikalisch zwischen 0 und 100 %; −100 % ist ein Fehlercode des Sensors.

**Kontrollwert (Voranalyse, Pflichtenheft R6):** 31 ungültig, 12 fehlend, 1.718 ok

**Erwartung laut Definition:** Relative Luftfeuchte liegt physikalisch zwischen 0 und 100 %; −100 % ist ein Fehlercode.

In [ ]:
# --- R6: Luftfeuchte außerhalb des gültigen Bereichs (konfiguration/sensoren.csv, 0–100 %) als ungültig markieren ---
h_min, h_max = REGELN.loc["humidity", "min"], REGELN.loc["humidity", "max"]   # gültiger Bereich in %
ungueltig = ~q["humidity"]["humidity"].between(h_min, h_max) & q["humidity"]["humidity"].notna()  # außerhalb, aber nicht fehlend
q["humidity"].loc[ungueltig, "qualitaet"] = f"ungueltig: außerhalb {h_min:g}–{h_max:g} %"  # Grund vermerken
q["humidity"].loc[ungueltig, "humidity"] = float("nan")             # Messwert leeren
q["humidity"]["qualitaet"].value_counts()                           # Kontrolle


**Ergebnis:**

humidity: 1.718 ok · **31 ungültig (< 0 %)** · 12 fehlend – wie erwartet (R6).

## 5. temperature – Frostwerte behalten (R7)

**Regel:** Werte < 0 °C bleiben erhalten, werden aber gekennzeichnet

**Warum?** Frost im Winter ist real; Löschen würde echte Daten vernichten.

**Kontrollwert (Voranalyse, Pflichtenheft R7):** 4.033 „ok: Frost (plausibel)“, 2.032 fehlend, 247.365 ok

**Erwartung laut Recherche:** Im Sommer halten Bienen das Brutnest bei ca. 33–36 °C (Mittel ca. 35,5 °C) [T1][T2]; im Winter liegt der Kern der Wintertraube bei ca. 20–30 °C, der Rand bei ca. 12 °C [T2][T3]. Werte unter 0 °C sind daher nur möglich, wenn der Sensor außerhalb der Traube misst (Rand oder leerer Stockbereich) – deshalb behalten statt löschen (R7).

In [ ]:
# --- R7a: physikalisch unmögliche Temperaturen (konfiguration/sensoren.csv, z. B. −227 °C) als ungültig markieren ---
t_min, t_max = REGELN.loc["temperature", "min"], REGELN.loc["temperature", "max"]   # gültiger Bereich in °C
t = q["temperature"]                                                # kurzer Name
ungueltig = ~t["temperature"].between(t_min, t_max) & t["temperature"].notna()      # außerhalb, aber nicht fehlend
t.loc[ungueltig, "qualitaet"] = f"ungueltig: außerhalb {t_min:g}–{t_max:g} °C"      # Grund vermerken
t.loc[ungueltig, "temperature"] = float("nan")                      # Messwert leeren
# --- R7: Frostwerte kennzeichnen, aber behalten ---
frost = t["temperature"] < 0                                        # True, wo die Temperatur unter 0 °C liegt (gültig)
t.loc[frost, "qualitaet"] = "ok: Frost (plausibel)"                 # nur kennzeichnen – Wert bleibt erhalten
t["qualitaet"].value_counts()                                       # Kontrolle


**Ergebnis:**

temperature: 247.365 ok · **4.033 Frost (plausibel, behalten)** · 2.032 fehlend – wie erwartet (R7).

## 6. flow – Ausflug und Einflug trennen (R8)

**Regel:** Die zwei untereinander gestapelten Messreihen werden zu zwei Spalten `flow_aus` und `flow_ein` je Minute

**Warum?** Gemischt ergeben Summen und Mittelwerte keinen Sinn.

**Kontrollwert (Voranalyse, Pflichtenheft R8):** Reihe 0 (Ausflug) −631…0 und Reihe 1 (Einflug) 0…7.999 mit je 1.256.918 Zeilen; nach dem Zusammenführen 1.256.918 Minuten, alle „both“ (beide Werte vorhanden)

In [ ]:
# --- R8: flow in Ausflug (Reihe 0) und Einflug (Reihe 1) trennen ---
f = q["flow"]                                                            # kurzer Name für die flow-Tabelle
reihe = (f["zeit_utc"].diff() < pd.Timedelta(hours=-2)).cumsum()         # Zeitsprung zurück = 2. Messreihe beginnt
print(f.groupby(reihe)["flow"].agg(["min", "max", "count"]))            # Kontrolle: Wertebereich + Anzahl je Reihe
aus = f[reihe == 0][["zeit_utc", "timestamp", "flow"]].rename(columns={"flow": "flow_aus"})  # Reihe 0 = Ausflug
ein = f[reihe == 1][["zeit_utc", "flow"]].rename(columns={"flow": "flow_ein"})              # Reihe 1 = Einflug
q["flow"] = aus.merge(ein, on="zeit_utc", how="outer", indicator=True)   # je Minute nebeneinanderstellen
print(q["flow"]["_merge"].value_counts())                                # Kontrolle: "both" = beide Werte vorhanden
q["flow"] = q["flow"].drop(columns="_merge")                             # Hilfsspalte wieder entfernen
q["flow"]["qualitaet"] = "ok"                                            # Qualitätsspalte (flow hatte keine Lücken)

**Ergebnis:**

Reihe 0 (Ausflug) −631…0 und Reihe 1 (Einflug) 0…7.999 mit je 1.256.918 Zeilen.

Zusammengeführt: 1.256.918 Minuten, **alle „both“** – beide Messreihen passen exakt 1:1 zusammen (R8).

## 6a. Ausreißer markieren – robuster rollierender z-Score (R9)

**Regel:** |z| > 3 im gleitenden Fenster → neue Spalte `<messwert>_auffaellig` = True; der Messwert bleibt erhalten (markieren statt löschen).

**Warum?** Eine feste Grenze passt nicht zu Tag/Nacht und Sommer/Winter (nachts ist flow 150 ein Ausreißer, mittags normal). Der gleitende Median mit MAD vergleicht jeden Wert nur mit seiner Umgebung und wird von Ausreißern selbst kaum verschoben (Methode aus dem Unterricht, gleiche Fenstergrößen).

**Erwartung laut Statistik:** Bei normalverteilten Werten liegen nur ca. 0,3 % außerhalb von ±3 Standardabweichungen (68-95-99,7-Regel); Sensordaten mit Tagesgang und Sprüngen lassen mehr Markierungen erwarten.

**Kontrollwert (Voranalyse, Befund E8):** temperature ca. 5.800 · humidity ca. 130 · weight ca. 60 · flow_aus ca. 8.900 · flow_ein ca. 15.900 auffällige Werte

In [ ]:
# --- R9: Ausreißer markieren mit robustem rollierendem z-Score (Methode aus dem Unterricht) ---
def robuster_z(werte, fenster):
    """Robuster z-Score: Abstand zum gleitenden Median geteilt durch 1,4826 × MAD."""
    median = werte.rolling(fenster, center=True, min_periods=1).median()                # gleitender Median = "normaler" Wert im Fenster
    mad = (werte - median).abs().rolling(fenster, center=True, min_periods=1).median()  # MAD = typische Abweichung vom Median im Fenster
    return (werte - median) / (1.4826 * mad).replace(0, float("nan"))                   # 1,4826 × MAD ≈ Standardabweichung; MAD = 0 → keine Aussage

# (Sensor, Spalte, Fenster in Messwerten) – Fenstergrößen wie im Unterricht
pruefung = [("temperature", "temperature", 60),   # 60 × 5 Min  = 5 Stunden
            ("humidity",    "humidity",     5),   # 5 × 12 Std  = 2,5 Tage
            ("weight",      "weight",      14),   # 14 × 12 Std = 1 Woche
            ("flow",        "flow_aus",    60),   # 60 × 1 Min  = 1 Stunde
            ("flow",        "flow_ein",    60)]   # 60 × 1 Min  = 1 Stunde

ergebnis = []                                                                   # Sammlung für die Kontrolltabelle
for s, spalte, fenster in pruefung:                                             # für jede zu prüfende Spalte ...
    t = q[s]                                                                    # ... kurzer Name für die Tabelle
    werte = t[spalte].where(t["qualitaet"].str.startswith("ok"))                # ... nur gültige Werte prüfen (ungültig/fehlend bleiben außen vor)
    if s == "flow":                                                             # ... bei flow:
        werte = werte.where(werte != 0)                                         #     0 = kein Bienenflug → nicht mitrechnen (wie im Unterricht)
    z = robuster_z(werte.dropna(), fenster).reindex(t.index)                    # ... z-Score berechnen und allen Zeilen zuordnen
    t[f"{spalte}_auffaellig"] = z.abs() > 3                                     # ... neue Spalte True/False – Messwert bleibt erhalten (markieren statt löschen)
    anzahl = int(t[f"{spalte}_auffaellig"].sum())                               # ... Anzahl auffälliger Werte
    ergebnis.append({"spalte": spalte, "fenster": fenster, "geprueft": werte.notna().sum(),
                     "auffaellig": anzahl, "anteil_%": round(anzahl / werte.notna().sum() * 100, 2)})
pd.DataFrame(ergebnis)                                                          # Kontrolle: Übersicht je Spalte

**Ergebnis:**

Auffällig (|z| > 3): temperature **5.762** (2,3 %) · humidity **134** (7,8 %) · weight **60** (3,6 %) · flow_aus **8.898** (1,6 %) · flow_ein **15.932** (3,0 %).

humidity ist mit 7,8 % hoch: Das Fenster umfasst nur 5 Messwerte (2,5 Tage), ein Wetterwechsel reicht schon für |z| > 3 → **auffällig heißt nicht falsch**; Werte bleiben erhalten und werden in der Analyse bewusst ein- oder ausgeschlossen.

**So liest man den z-Score (zwei Beispiele aus den Daten):**

| Zeit (Ortszeit) | Wert | gleitender Median | z | Deutung |
| :-- | --: | --: | --: | :-- |
| 05.08.2018 01:25 | flow_ein 7.999 | 12 | **1.077** | Nachbarn 6–29 Bienen/Min → 7.999 ist ein Sensor-Fehlercode → echter Ausreißer |
| 08.01.2018 09:30 | temperature −1,59 °C | −1,5 °C | **−3,9** | Abweichung nur 0,09 °C; Werte sind so gleichmäßig, dass schon Rauschen auffällt → kein echter Ausreißer |

z = (Wert − Median) / (1,4826 × MAD) = „Wie viele typische Schwankungen liegt der Wert von seinen Nachbarn entfernt?“ Ab 3 gilt er als auffällig.

temperature: nur **1.773** der 5.762 auffälligen Werte weichen mehr als 1 °C ab (Befund E9) → eine zusätzliche Mindestabweichung wird geprüft (Entscheidung D2).

## 6b. Kontrolle: auffällige flow-Werte je Uhrzeit

**Warum?** Prüfen, ob die Methode zu jeder Tageszeit gleich gut passt (z. B. Dämmerung, Nacht, Mittagshitze).

**Eigene Erwartung:** Flüge mit Höhepunkt am Nachmittag; nachts mehr auffällige Werte (wenig Flug – kleine Schwankungen wirken groß)

**Erwartung laut Recherche:** Die Flugaktivität folgt vor allem Temperatur und Sonnenstrahlung (in einer Feldstudie zusammen 78 % der Schwankung) [T4] – beide sind am frühen Nachmittag am höchsten.

In [ ]:
# --- Kontrolle: Anteil auffälliger flow-Werte je Uhrzeit (Ortszeit) ---
fl = q["flow"]                                                            # kurzer Name für die flow-Tabelle
stunde = fl["zeit_utc"].dt.tz_convert(ZEITZONE).dt.hour                    # UTC → Ortszeit des Standorts, nur die Stunde (0–23)
je_stunde = pd.DataFrame({
    "fluege_ein": fl.groupby(stunde)["flow_ein"].sum(),                   # Summe Einflüge je Stunde (zeigt Tag/Nacht-Rhythmus)
    "auffaellig_%": (fl.groupby(stunde)["flow_ein_auffaellig"].sum()      # auffällige Werte je Stunde ...
                     / fl[fl["flow_ein"] != 0].groupby(stunde)["flow_ein"].size()  # ... geteilt durch geprüfte Werte (≠ 0)
                     * 100).round(1),                                     # ... in Prozent
})
je_stunde.index.name = "stunde_ortszeit"                                  # sprechender Name für die Index-Spalte
je_stunde.T                                                               # quer anzeigen: 24 Stunden nebeneinander

**Ergebnis:**

Die meisten Einflüge gibt es um **14–15 Uhr**; eine Mittagsdelle ist im Durchschnitt nicht zu sehen.

Anteil auffälliger Werte: tagsüber (5–19 Uhr) ca. **2 %**, nachts (22–1 Uhr) **5–6,5 %** → nachts gibt es wenige, kleine Werte, dadurch fällt schon ein kleiner Ausschlag auf. Die Methode ist in Nacht/Dämmerung **zu streng** (Befund **E8**).

→ Deshalb nur markieren; Sonnenauf-/-untergang und Wetter (N3) helfen in der Analyse (N5), echte Ausreißer von Tag/Nacht-Wechseln zu unterscheiden.

## 6c. Sensitivitätsanalyse (Empfindlichkeitsprüfung) D2 – Mindestabweichung temperature

**Frage:** Ab welcher Abweichung vom gleitenden Median ist ein auffälliger Temperaturwert ein echter Ausreißer und kein Rauschen?

**Vorgehen:** Startwert **1 °C**, verglichen mit einem niedrigeren (**0,5 °C**) und einem höheren Wert (**2 °C**). Nur Auswertung und Visualisierung – die Markierung aus 6a bleibt unverändert, bis D2 entschieden ist (Entscheidungszyklus: visualisieren → Prüfung Mensch + KI → Entscheidung).

**Eigene Erwartung:** Je höher die Mindestabweichung, desto weniger Werte bleiben auffällig.

**Kontrollwert (Befund E9):** bei 1 °C ca. 1.770 von 5.762.

In [ ]:
# --- D2: Wie viele auffällige Werte bleiben bei 0,5 · 1 · 2 °C Mindestabweichung? ---
te = q["temperature"]                                                         # kurzer Name für die Temperatur-Tabelle
gueltig = te["temperature"].where(te["qualitaet"].str.startswith("ok")).dropna()  # nur gültige Werte (wie in 6a)
median_te = gueltig.rolling(60, center=True, min_periods=1).median().reindex(te.index)  # gleitender Median (Fenster 60 = 5 Std)
abweichung = (te["temperature"] - median_te).abs()                            # Abstand zum Median in °C (nur Hilfsgröße, wird nicht gespeichert)
auff = te["temperature_auffaellig"]                                           # Markierung aus 6a (|z| > 3)

stufen = [0.5, 1.0, 2.0]                                                      # Startwert 1 °C, ein niedrigerer und ein höherer Vergleichswert
d2 = pd.DataFrame([{"mindestabweichung_°C": g,                                                    # geprüfte Stufe
                    "bleiben_auffaellig": int((auff & (abweichung > g)).sum()),                   # |z| > 3 UND Abweichung > Stufe
                    "gelten_als_rauschen": int((auff & (abweichung <= g)).sum())}                 # |z| > 3, aber Abweichung klein
                   for g in stufen])
d2["anteil_bleiben_%"] = (d2["bleiben_auffaellig"] / int(auff.sum()) * 100).round(1)  # Anteil an allen 5.762 Markierungen
d2                                                                            # Kontrolle: Vergleichstabelle

In [ ]:
# --- D2 visualisieren (Grafik-Standard: erst Überblick, dann zwei begründete Beispiele) ---
import matplotlib.pyplot as plt                                               # Diagramme
import matplotlib.dates as mdates                                             # Uhrzeit-Format der x-Achse

lokal = te["zeit_utc"].dt.tz_convert(ZEITZONE)                                 # Zeitachse in Ortszeit des Standorts
echt = auff & (abweichung > 1.0)                                              # auffällig UND mehr als 1 °C Abweichung
rauschen = auff & (abweichung <= 1.0)                                         # auffällig, aber höchstens 1 °C Abweichung

fig, ax = plt.subplots(1, 3, figsize=(20, 5.2), gridspec_kw={"width_ratios": [1.4, 1, 1]})
fig.suptitle("Temperatur im Bienenstock: Zwei Drittel der markierten Werte sind nur Rauschen (Abweichung ≤ 1 °C)",
             fontsize=14, fontweight="bold")

# (1) Überblick: markierte Werte je Monat – Rauschen vs. echte Sprünge
monat = lokal.dt.to_period("M")
uebersicht = pd.DataFrame({"Rauschen (≤ 1 °C)": rauschen.groupby(monat).sum(),
                           "echter Sprung (> 1 °C)": echt.groupby(monat).sum()})
x = range(len(uebersicht))
ax[0].bar(x, uebersicht["Rauschen (≤ 1 °C)"], color="lightgrey", edgecolor="grey", label=f"Rauschen (≤ 1 °C): {int(rauschen.sum()):,}".replace(",", "."))
ax[0].bar(x, uebersicht["echter Sprung (> 1 °C)"], bottom=uebersicht["Rauschen (≤ 1 °C)"], color="tab:red",
          label=f"echter Sprung (> 1 °C): {int(echt.sum()):,}".replace(",", "."))
ax[0].set_xticks(list(x)[::3], [str(p) for p in uebersicht.index][::3], rotation=45)
ax[0].set(title="Überblick: markierte Werte je Monat (|z| > 3)", xlabel="Monat", ylabel="Anzahl markierter Messwerte (5-Min.-Werte)")
ax[0].legend(title="Entscheidung D2 = 1 °C")

# (2) + (3) Beispiele mit Begründung im Titel
beispiele = [(ax[1], "2018-01-08", "Beispiel Rauschen – ruhiger Wintertag 08.01.2018:\nmarkiert, obwohl nur 0,1 °C neben dem Normalwert"),
             (ax[2], str(lokal[abweichung[auff].idxmax()].date()), "Beispiel echter Sprung – größte Abweichung im Datensatz:\nkurze Spitze weit über dem Normalwert")]
for a, datum, titel in beispiele:
    t = lokal.dt.date == pd.Timestamp(datum).date()                            # nur dieser Tag
    a.plot(lokal[t], te.loc[t, "temperature"], color="steelblue", lw=1.2, label="Messwert")
    a.plot(lokal[t], median_te[t], color="black", lw=1, ls=":", label="Normalwert (gleitender Median, 5 Std.)")
    a.scatter(lokal[t & rauschen], te.loc[t & rauschen, "temperature"], color="lightgrey", edgecolors="grey", s=35, zorder=3, label="markiert: Rauschen")
    a.scatter(lokal[t & echt], te.loc[t & echt, "temperature"], color="tab:red", edgecolors="black", s=45, zorder=3, label="markiert: echter Sprung")
    i = abweichung[t & auff].idxmax()                                          # stärkster markierter Punkt des Tages
    text = f"Messwert {te.loc[i, 'temperature']:.2f} °C\nNormalwert {median_te[i]:.2f} °C\nAbweichung {abweichung[i]:.2f} °C"
    a.annotate(text.replace(".", ","),                                         # deutsches Dezimalkomma
               xy=(lokal[i], te.loc[i, "temperature"]), xytext=(0.03, 0.97), textcoords="axes fraction", va="top", fontsize=9,
               bbox=dict(boxstyle="round", fc="white", ec="grey"), arrowprops=dict(arrowstyle="->", color="grey"))
    a.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=ZEITZONE))
    a.set(title=titel, xlabel="Uhrzeit (Ortszeit)", ylabel="Temperatur im Stock (°C)")
    a.legend(fontsize=8, loc="lower right")

fig.text(0.01, -0.02, f"Quelle: Bee Haven {ORT['name']}, Temperatursensor im Stock (5-Min.-Werte, 01/2017–05/2019). "
         "Markiert = robuster z-Score |z| > 3 im Fenster von 5 Stunden. Normalwert = gleitender Median.", fontsize=9, color="grey")
plt.tight_layout()
bild_speichern(plt.gcf(), "d2_sensitivitaet_temperatur.png", dpi=120, bbox_inches="tight")
plt.show()

**Ergebnis:**

| Mindestabweichung | bleiben auffällig | gelten als Rauschen | Anteil bleiben |
| :-- | --: | --: | --: |
| 0,5 °C | 2.624 | 3.138 | 45,5 % |
| **1 °C (Startwert)** | **1.773** | **3.989** | **30,8 %** |
| 2 °C | 921 | 4.841 | 16,0 % |

- **Links (Überblick):** Je Monat grau = Rauschen, rot = echter Sprung. Echte Sprünge häufen sich im **Winter (Nov.–März)**, wenn die Stocktemperatur nicht mehr auf ca. 35 °C geregelt wird (Wintertraube) – Hinweis für Warnregel P5. 3.138 der 5.762 Markierungen (54 %) weichen höchstens 0,5 °C vom Median ab, 1.115 sogar nur ≤ 0,1 °C → überwiegend Rauschen bzw. Messgenauigkeit.
- **Mitte (ruhiger Wintertag):** Die grauen Punkte liegen auf einer glatten Kurve. Am Tiefpunkt (09–11 Uhr) „schneidet“ der gleitende Median die Spitze ab – die Markierungen sind ein Effekt der Methode, keine Ausreißer.
- **Rechts (unruhiger Tag):** Nur die Spitzen mit 26–28 °C (Abweichung > 2 °C) werden markiert. Schwankungen von 5–8 °C bleiben unmarkiert, weil die typische Schwankung (MAD) an diesem Tag selbst groß ist → Gegenstück zum ruhigen Tag.

**Einordnung:** Der z-Score allein ist bei gleichmäßigen Werten zu empfindlich; eine Mindestabweichung trennt Rauschen von echten Sprüngen. 1 °C halbiert die Markierungen gegenüber 0,5 °C; 2 °C behält nur deutliche Sprünge. **Entscheidung D2 (28.09.2026): 1 °C** – ausgewogen; umgesetzt in Kapitel 6d.


## 6d. Entscheidung D2 umsetzen – Mindestabweichung 1 °C für temperature

**Regel (R9, ergänzt):** temperature gilt nur als auffällig, wenn |z| > 3 **und** die Abweichung vom gleitenden Median mehr als 1 °C beträgt.

**Warum?** Sensitivitätsanalyse 6c: 0,5 °C behält viel Rauschen, 2 °C verliert kleinere echte Sprünge; 1 °C ist ausgewogen (Entscheidung nach Prüfung Mensch + KI).

**Kontrollwert (Kap. 6c):** temperature 5.762 → 1.773 auffällige Werte

In [ ]:
# --- D2: Mindestabweichung 1 °C anwenden (nur temperature) ---
MINDESTABWEICHUNG_C = 1.0                                                     # Entscheidung D2 vom 28.09.2026
vorher = int(te["temperature_auffaellig"].sum())                              # Anzahl vor der Anpassung
te["temperature_auffaellig"] = te["temperature_auffaellig"] & (abweichung > MINDESTABWEICHUNG_C)  # beide Bedingungen
print(f"temperature auffällig: vorher {vorher:,} → nachher {int(te['temperature_auffaellig'].sum()):,}")

**Ergebnis:**

temperature: 5.762 → **1.773** auffällige Werte. Die übrigen 3.989 Markierungen waren Rauschen und sind aufgehoben; Messwerte bleiben unverändert.

## 7. Speichern nach silver/cleaned

**Wohin?** `cleaned/schwartau/<sensor>/<sensor>_schwartau_cleaned__<UTC-Zeitstempel>_UTC.csv` – jeder Lauf erzeugt eine neue Version (Rückgriff auf den letzten guten Stand); `processing` wird nie verändert.

**Kontrollwert:** 4 neue Dateien + Protokoll mit Zeilen und Qualitätsstufen je Sensor

In [ ]:
# --- Bereinigte Daten mit UTC-Zeitstempel speichern ---
stempel = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")    # aktueller Zeitpunkt in UTC (HH = 24 Stunden)
protokoll = []                                                       # leeres Protokoll
for s in sensoren:                                                   # für jeden Sensor ...
    ordner = f"{CLEANED}/{s}"                                        # ... Zielordner cleaned/<stadt>/<sensor>/
    ordner_anlegen(ordner)                                           # ... anlegen, falls nicht vorhanden
    ziel = f"{ordner}/{s}_{STADT}_cleaned__{stempel}_UTC.parquet"        # ... Dateiname mit Zeitstempel
    q[s].to_parquet(ziel, index=False, storage_options=SO)           # ... als Parquet speichern (neue Version, nichts wird überschrieben)
    for alt, _ in dateien(ordner, f"{s}_{STADT}_cleaned__*_UTC.parquet"):  # ... ältere Versionen ins _archiv des Ordners
        if alt.rsplit("/", 1)[-1] != ziel.rsplit("/", 1)[-1]:
            archivieren(alt)
    protokoll.append({"sensor": s, "zeilen": len(q[s]),              # ... Protokollzeile: Zeilen +
                      **q[s]["qualitaet"].value_counts().to_dict()}) #     Anzahl je Qualitätsstufe
    print("gespeichert:", ziel.replace(WURZEL, ""))                  # ... Kontrolle
pd.DataFrame(protokoll).fillna(0)                                    # Protokoll als Tabelle anzeigen

**Ergebnis:**

4 Dateien in `silver/cleaned/schwartau/<sensor>/…_cleaned__<UTC-Zeitstempel>_UTC.csv (neue Version je Lauf)` gespeichert.

flow hat jetzt 1.256.918 Zeilen (je Minute `flow_aus` + `flow_ein`) statt 2.513.836 gestapelter Zeilen.

Hinweis: Die bereinigten CSV sind größer als die Rohdaten (flow ca. 68 MB), weil `zeit_utc`, `qualitaet` und die `_auffaellig`-Spalten dazukommen – für Gold/Power BI später Parquet prüfen.

## Zusammenfassung N2

| Sensor | Zeilen | ok | ungültig | fehlend | auffällig (R9) | Hinweis |
| :-- | --: | --: | --: | --: | --: | :-- |
| flow | 1.256.918 | 1.256.918 | 0 | 0 | aus 8.898 · ein 15.932 | Ausflug/Einflug getrennt (R8) |
| humidity | 1.761 | 1.718 | 31 | 12 | 134 | < 0 % (R6) |
| temperature | 253.430 | 251.398 | 0 | 2.032 | 1.773 (D2: > 1 °C) | davon 4.033 Frost, behalten (R7) |
| weight | 1.761 | 1.619 | 130 | 12 | 59 | Zeitraum „Stock nicht auf Waage“ 28.04.–02.07.2018 (R5a, E5); darin alle 102 Werte < 20 kg (R5) |

Alle Kontrollwerte erfüllt.

**Quellen zu den Erwartungen (Recherche 29.09.2026):**
- [T1] Fahrenholz, Lamprecht, Kraus: „Thermal investigations of a honey bee colony: thermoregulation of the hive during summer and winter“, Journal of Comparative Physiology B (1989): https://link.springer.com/article/10.1007/BF00694379
- [T2] Stabentheiner, Kovac, Brodschneider: „Honeybee Colony Thermoregulation – Regulatory Mechanisms and Contribution of Individuals …“, PLOS ONE (2010): https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2813292/
- [T3] Wikipedia, „Winter cluster“ (Kern ca. 20–30 °C, Rand ca. 12 °C; Übersicht mit Einzelbelegen): https://en.wikipedia.org/wiki/Winter_cluster
- [T4] „Predictive modelling of honey bee foraging activity using local weather conditions“, Apidologie (2018): https://link.springer.com/content/pdf/10.1007/s13592-018-0565-3.pdf
- Statistik: 68-95-99,7-Regel der Normalverteilung (Standardwissen Statistik).

Hinweis: „Kontrollwert“ = Zahl aus Voranalyse oder Vorschritt, die übereinstimmen **muss**; „Erwartung laut Recherche“ = unabhängige Quelle, Abweichungen sind möglich und werden benannt. Auffällige Werte nachts häufiger (E8). Nächster Schritt: N3 Wetter (Bright Sky/DWD + Sonnenstand), danach N4 Gold und N5 Analyse. Lücken werden nicht aufgefüllt (P6 = c, Pflichtenheft Schritt 3.1).